# 02 — Preprocessing Pipeline

This notebook validates and benchmarks the full preprocessing pipeline:

1. Ben Graham preprocessing — parameter sweep (sigma: 5, 10, 20)
2. Augmentation pipeline — visual verification
3. Batch preprocessing — time & quality check
4. DataLoader construction — verify shapes & normalization

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

from src.data.preprocessing import BenGrahamPreprocessor
from src.data.augmentation import get_train_transforms, get_val_transforms
from src.data.dataset import APTOSDataset
from src.utils.config import load_config

cfg = load_config('../configs/default.yaml')
print('Config loaded')

## 1. Ben Graham Sigma Sweep

In [ ]:
TRAIN_IMG = Path('../data/raw/aptos2019/train_images/train_images')
TRAIN_CSV = Path('../data/raw/aptos2019/train_split.csv')
train = pd.read_csv(TRAIN_CSV)

sample_path = TRAIN_IMG / f"{train.iloc[100]['id_code']}.png"
img = cv2.cvtColor(cv2.imread(str(sample_path)), cv2.COLOR_BGR2RGB)

sigmas = [5, 10, 20]
fig, axes = plt.subplots(1, len(sigmas) + 1, figsize=(18, 4))
axes[0].imshow(cv2.resize(img, (224, 224)))
axes[0].set_title('Original (resized)', fontweight='bold')
axes[0].axis('off')

for ax, sigma in zip(axes[1:], sigmas):
    preprocessor = BenGrahamPreprocessor(224, sigma=sigma)
    ax.imshow(preprocessor(img))
    ax.set_title(f'sigma={sigma}', fontweight='bold')
    ax.axis('off')

fig.suptitle('Ben Graham σ Sweep', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/preprocessing_sigma_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('sigma=10 is the literature default (Ben Graham, 2015)')

## 2. Augmentation Pipeline Verification

Visual confirmation that augmentations are reasonable for retinal images.

In [ ]:
transform = get_train_transforms(224)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Training Augmentation Samples', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    augmented = transform(image=img)
    img_t = augmented['image']
    # Denormalise for display
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_display = img_t.permute(1,2,0).numpy() * std + mean
    img_display = np.clip(img_display, 0, 1)
    ax.imshow(img_display)
    ax.axis('off')
    ax.set_title(f'Aug {i+1}', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. DataLoader Shape Verification

In [ ]:
from src.data.dataset import create_data_loaders

train_loader, val_loader, test_loader = create_data_loaders(
    train_csv='../data/raw/aptos2019/train_split.csv',
    val_csv='../data/raw/aptos2019/val_split.csv',
    test_csv='../data/raw/aptos2019/test_split.csv',
    train_image_dir='../data/raw/aptos2019/train_images/train_images',
    val_image_dir='../data/raw/aptos2019/val_images/val_images',
    test_image_dir='../data/raw/aptos2019/test_images/test_images',
    train_transform=get_train_transforms(224),
    val_transform=get_val_transforms(224),
    image_size=224,
    batch_size=8,
)

images, labels = next(iter(train_loader))
print(f'Image batch shape: {images.shape}')
print(f'Label batch shape: {labels.shape}')
print(f'Pixel range: [{images.min():.3f}, {images.max():.3f}]')
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')